# Гипотеза 1

## Какие проблемы вам нужно решить: вы должны сгенерировать как минимум 50 новых признаков (название новых признаков должно начинаться с `f_`)

- Новые признаки не должны иметь пропущенных значений
- Категориальные признаки должны быть преобразованы в числовые
- Вы должны применить как минимум пять разных преобразователей

### Ваши действия следующие:

1. Переименуйте этот ноутбук из `H1.ipynb` в `1_Name_Surname.ipynb`
2. Проанализируйте сырые данные и подготовьте текстовое описание того, какие признаки и как вы будете преобразовывать
3. Создайте новые признаки (используйте преобразователи данных с лекции) для ВСЕГО набора данных (оставьте исходные признаки без изменений).
4. Новые признаки должны начинаться с префикса `f_`
5. Напишите `Description` о том, какие проблемы вы нашли в данных и как вы предобработали данные
6. Отправьте этот ноутбук и предобработанный тестовый датасет в личном сообщении в телеграм.

## **ВАЖНО!!! `Name_Surname` не должен изменяться в других гипотезах**


## **ВАЖНО!!! Предобработанный тестовый датасет должен иметь `1_Name_Surname.csv`**

# Description (Анализ сырых данных)
Размер датасета - 30 471 строк × 62 колонки.

Разбиение задано столбцом dtype -

 1) train: 20 483 строк

 2) test: 9 988 строк

Типы признаков -

 1) числовых колонок: 41

 2) категориальных (object): 21

служебные: dtype, __price, __churn

Пропуски (missing values) - в исходных данных пропуски есть в ряде числовых колонок. Самые проблемные (доля NaN, %)

 1) build_year ≈ 44.66%

 2) state ≈ 44.34%

 3) total_trans_amt ≈ 40.61%

 4) 0_17_all ≈ 40.61%

 5) max_floor ≈ 31.31%

 6) num_room ≈ 31.31%

 7) cafe_sum_1000_min_price_avg ≈ 21.47%

 8) life_sq ≈ 20.78%
и ещё несколько с меньшей долей пропусков

В категориях много бинарных категориальных признаков 0/1 , но в object, а так же есть высококардинальная колонка timestamp (~1151 уникальное значение).

Общие принципы -
 1) исходные признаки не меняются все околонки остаются как есть.
 2) создаем новые признаки с префиксом f_ и добавляем их в new_dataset.
 3) все новые признаки строим для всего датасета и train, и test вместе, но параметры медианы, квантили, частоты считаем только на train.
 4) в новых f_ признаках не должно быть NaN, поэтому делаем иммутацию.

 Числовые признаки все numeric, кроме __price, __churn, для каждого числового признака X я делаем набор новых признаков -
1) Индикатор пропуска f_isna_X = 1, если исходно было NaN, иначе 0.
2) Заполнение пропусков медианой median imputation, по train, берем median X_train и подставляем вместо NaN, чтобы получить стабильные значения без удаления строк.
3) Обрезка выбросов по квантилям quantile clipping, по train, обрезаем значения по границам 1% и 99% квантилей train - f_clip_X = clip(X_imp, q01_train, q99_train), чтобы снизить влияние экстремальных значений.
4) Robust scaling по train, центрирование по медиане и масштабирование по IQR (Q3–Q1) - f_rs_X = (f_clip_X − median_train) / IQR_train, чтобы привести признаки к сопоставимому масштабу и быть устойчивой к выбросам.
5) Логарифмирование log1p для неотрицательных, после clipping f_log1p_X = log(1 + f_clip_X), чтобы сжать длинные хвосты и сделать распределения ближе к нормальным.

Категориальные признаки object, кроме dtype -
1) Frequency encoding (по train) f_freq_C = частота категории в train доля вероятности, чтобы превратить категорию в число без большой размерности.
2) One-hot encoding для top k категорий (по train) для частых уровней создаю бинарные колонки f_ohe_C__level = 1/0

Использованные преобразователи -
1) Missing indicator → f_isna_*
2) Median imputation по типу SimpleImputer(strategy="median")
3) Quantile clipping
4) Robust scaling
5) Log transform
6) Categorical encoding - frequency encoding (f_freq_*) и one-hot encoding (f_ohe_*)

Найденные проблемы в данных -
1) В ряде числовых признаков присутствуют пропуски, в некоторых колонках доля NaN превышает 30–40%.
2) Числовые признаки имеют разный масштаб, встречаются выбросы,асимметричные распределения.
3) В данных есть категориальные признаки, которые нельзя использовать  без преобразования их в числовой формат.

Что было сделано (Технические признаки) -
1) для каждого числового признака кроме __price и __churn были построены -

     * индикатор пропуска f_isna_*,

     * заполнение пропусков медианой train,

     * обрезка выбросов по квантилям train (1%–99%) f_clip_*,

     * устойчивое масштабирование по медиане и IQR train f_rs_*,

     * логарифмирование log1p для уменьшения асимметрии f_log1p_*.

2) Для категориальных признаков выполнено числовое кодирование -

     * frequency encoding по train f_freq_*,

     * one-hot encoding для top-K категорий f_ohe_*__*.

В результате получено 220 новых признаков f_ и в новых признаках нет пропусков, все они числовые. Количество строк сохранено без изменений (30 471).

Разбивка -

39 f_isna_* - индикаторы пропусков

39 f_clip_* - числовые после clipping

39 f_rs_* - robust scaled

39 f_log1p_* - логарифмированные

20 f_freq_* - frequency encoding категорий

44 f_ohe_* - one-hot для top-K категорий

Расшифровка признаков -
 1) f_isna_<X> индикатор пропуска в исходной колонке <X>
      * 1 = в исходной колонке <X> был NaN

      * 0 = значение было заполнено

2) f_clip_<X> числовой признак <X> после заполнения пропуска медианой (по train) и обрезки выбросов по квантилям train (1% и 99%).

3)f_rs_<X> -  f_clip_<X>, приведённый к устойчивому масштабу
через медиану и IQR = межквартильный размах, по train

0 ≈ около медианы train

+ значение выше типичного

- значение ниже типичного

Формула -



  






In [ ]:
from IPython.display import display, Math

display(Math(r'f_{rs,X} = \frac{f_{clip,X} - \mathrm{median}(X_{train})}{IQR(X_{train})}'))
display(Math(r'IQR(X_{train}) = Q_{0.75}(X_{train}) - Q_{0.25}(X_{train})'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

4) f_log1p_<X> - логарифм от очищенного признака (f_clip_<X>)

Формула -

In [ ]:
from IPython.display import display, Math

display(Math(r'f_{log1p,X} = \log(1 + f_{clip,X})'))

<IPython.core.display.Math object>

5) f_freq_<C> - частотное кодирование категориальной колонки <C> значение равняется доли категории в train (от 0 до 1)

6) f_ohe_<C>__<level> One-Hot признак индикатор конкретной категории

Смысловые признаки -

1) Недвижимость - метраж, комнаты, этажность, состояние, возраст

f_sem_life_share = life_sq / full_sq
Доля жилой площади в общей

f_sem_non_life_sq = full_sq - life_sq
Нежилая площадь (кухня, коридоры)

f_sem_non_life_share = (full_sq - life_sq) / full_sq
Доля нежилой площади

f_sem_sqm_per_room = full_sq / num_room
Площадь на комнату

f_sem_rooms_per_10sqm = num_room / (full_sq/10)
Плотность комнат на 10 м²

f_sem_floor_ratio = floor / max_floor
Доля этажа от этажности

f_sem_floors_from_top = max_floor - floor
Сколько этажей до последнего

f_sem_is_first_floor = 1, если floor <= 1, иначе 0
Флаг первого этажа

f_sem_is_top_floor = 1, если floor == max_floor, иначе 0
Флаг последнего этажа

f_sem_is_low_floor = 1, если floor <= 2
Низкий этаж

f_sem_is_high_floor = 1, если floor/max_floor >= 0.8
Высокий этаж верхние 20%

f_sem_state_low = 1, если state <= 2
Состояние хуже, среднее”

f_sem_state_high = 1, если state >= 3
Состояние хорошее, отличное

f_sem_building_age = year(timestamp) - build_year (обрезано 0..200)
Возраст дома

f_sem_is_new_build = 1, если возраст <= 5
Новый дом

f_sem_is_old_build = 1, если возраст >= 50
Старый дом

2) Доступность, расстояния, транспорт

f_sem_access_total_km = mkad_km + metro_km_walk + railroad_station_walk_km + detention_facility_km
Суммарная удалённость

f_sem_access_score = 1/(1+mkad_km) + 1/(1+metro_km_walk) + 1/(1+rail_walk_km)
Скор доступности

f_sem_mkad_metro_ratio = mkad_km / metro_km_walk
Отношение удалённости от МКАД к удалённости от метро

f_sem_log_pt_walk = log1p(public_transport_station_min_walk)
Лог-время ходьбы до ОТ

3) Инфраструктура, плотности, кафе, материалы

f_sem_leisure_total = leisure_count_3000 + leisure_count_5000
Всего объектов досуга в радиусах 3км и 5км

f_sem_amenities_total = market + trc + sport + leisure3 + leisure5 + cafe_cnt + schools
Общий индекс удобств

f_sem_amenities_per_sqm = amenities_total / full_sq
Удобства на м² нормализация на размер жилья

f_sem_office_per_sqm = office_sqm_5000 / full_sq
Офисная площадь вокруг на м² квартиры

f_sem_child_density = 0_17_all / full_sq
Детская плотность на м²

f_sem_school_per_child = school_education_centers_top_20_raion / (0_17_all + 1)
Школы на ребёнка

f_sem_cafe_price_ratio_1500_1000 = cafe_sum_1500_min_price_avg / cafe_sum_1000_min_price_avg
Относительный уровень цен кафе (1500м vs 1000м).

f_sem_cafe_price_diff_1500_1000 = cafe1500 - cafe1000
Разница цен кафе (1500м vs 1000м).

f_sem_wood_share = build_count_wood / (build_count_wood + build_count_mix + 1)
Доля деревянной застройки по двум типам

f_sem_mix_share = build_count_mix / (build_count_wood + build_count_mix + 1)
Доля смешанной застройки

4) Риски, экология

f_sem_water_near = 1, если water_1line == "yes"
Вода рядом

f_sem_big_market = 1, если big_market_raion == "yes"
Крупный рынок в районе

f_sem_env_risk_count = сумма флагов (0/1) по:
oil_chemistry_raion, nuclear_reactor_raion, incineration_raion, radiation_raion, thermal_power_plant_raion
Сколько экологически опасных факторов

f_sem_noise_risk_count = big_road1_1line + railroad_1line
Сколько шумовых факторов

f_sem_culture_top25 = 1, если culture_objects_top_25 == "yes"
Культурные объекты top-25 рядом

f_sem_detention_raion = 1, если detention_facility_raion == "yes"
Наличие тюрьмы в районе (флаг).

f_sem_rail_terminal_raion = 1, если railroad_terminal_raion == "yes"
Ж/д терминал в районе (флаг).

f_sem_risk_index = 2*env_risk_count + 1*noise_risk_count + 1*detention_raion
Сводный индекс риска с весами

f_sem_risk_x_access = risk_index * access_total_km
Риск × удалённость (взаимодействие).

5) Банковские смысловые кредит, транзакции, активность

f_sem_used_credit = credit_limit - avg_open_to_buy (>=0)
Сколько кредита “использовано”.

f_sem_open_to_buy_ratio = avg_open_to_buy / credit_limit
Доля свободного лимита

f_sem_used_credit_ratio = used_credit / credit_limit
Доля использованного лимита

f_sem_revolving_ratio = total_revolving_bal / credit_limit
Revolving-остаток как доля лимита

f_sem_util_x_used_ratio = avg_utilization_ratio * used_credit_ratio
Взаимодействие утилизация × доля использования

f_sem_trans_avg_amount = total_trans_amt / total_trans_ct
Средний чек транзакции

f_sem_trans_amt_per_month = total_trans_amt / months_on_book
Объём транзакций в месяц

f_sem_trans_ct_per_month = total_trans_ct / months_on_book
Кол-во транзакций в месяц

f_sem_trans_amt_per_rel = total_trans_amt / total_relationship_count
Объём на отношение с банком.

f_sem_trans_ct_per_rel = total_trans_ct / total_relationship_count
Кол-во транзакций на отношение

f_sem_contacts_per_month = contacts_count_12_mon / months_on_book
Контакты с банком в месяц

f_sem_inactive_ratio = months_inactive_12_mon / months_on_book (0..1)
Доля неактивных месяцев

f_sem_active_ratio = 1 - inactive_ratio
Доля активных месяцев

f_sem_rel_per_age = total_relationship_count / (customer_age + 1)
Отношения с банком на возраст

f_sem_dep_ratio = dependent_count / (customer_age + 1)
Иждивенцы “на возраст

f_sem_change_interaction = total_amt_chng_q4_q1 * total_ct_chng_q4_q1
Взаимодействие динамики суммы и количества транзакций

f_sem_change_ratio = total_amt_chng_q4_q1 / total_ct_chng_q4_q1
Отношение изменение суммы к изменению количества

f_sem_util_low = 1, если avg_utilization_ratio < 0.2
Низкая утилизация

f_sem_util_high = 1, если avg_utilization_ratio > 0.8
Высокая утилизация

6) Время из timestamp сезонность, цикличность

f_sem_year = год из timestamp

f_sem_month = месяц (1..12)

f_sem_quarter = квартал (1..4)

f_sem_dayofweek = день недели (0=Mon..6=Sun)

f_sem_is_weekend = 1 если dayofweek >= 5

Циклические признаки

f_sem_month_sin = sin(2π*month/12)

f_sem_month_cos = cos(2π*month/12)

f_sem_dow_sin = sin(2π*dow/7)

f_sem_dow_cos = cos(2π*dow/7)

7) Ординальные ранги категорий u флаги

f_sem_income_rank — порядок дохода:
Less than $40K=0, $40–60K=1, $60–80K=2, $80–120K=3, $120K+=4 (Unknown=2)

f_sem_card_rank — уровень карты - Blue=0, Silver=1, Gold=2

f_sem_ecology_rank — экология: poor=0, satisfactory=1, no data=2, good=3, excellent=4

f_sem_education_rank — образование -  High School=0, Graduate=1 (Unknown=1)

f_sem_is_male — 1 если gender == 'M'

f_sem_is_married — 1 если marital_status == 'Married'

f_sem_is_investment — 1 если product_type == 'Investment'

f_int_* (все признаки)
1) Доход/карта × кредит и транзакции

f_int_income_x_used_ratio = income_rank * used_credit_ratio
→ высокий доход + высокая доля использования лимита

f_int_income_x_open_ratio = income_rank * open_to_buy_ratio
→ высокий доход + большая доля свободного лимита

f_int_income_x_trans_amt_m = income_rank * trans_amt_per_month
→ доход × объём трат в месяц

f_int_income_x_trans_ct_m = income_rank * trans_ct_per_month
→ доход × частота покупок

f_int_income_x_trans_avg = income_rank * trans_avg_amount
→ доход × средний чек

f_int_card_x_used_ratio = card_rank * used_credit_ratio
→ “статус карты” × использование лимита

f_int_card_x_trans_amt_m = card_rank * trans_amt_per_month
→ статус карты × траты в месяц

f_int_card_x_trans_ct_m = card_rank * trans_ct_per_month
→ статус карты × частота операций

f_int_high_util_and_high_income = 1 если util_high=1 и income_rank>=3
→ богатый клиент с высокой утилизацией лимита

f_int_low_util_and_low_income = 1 если util_low=1 и income_rank<=1
→ низкий доход + низкая утилизация (низкая активность).

2) Инфраструктура × доступность (ценность локации)

f_int_access_x_amen = access_score * amenities_total
→ удобно добираться × много инфраструктуры

f_int_access_x_amen_ps = access_score * amenities_per_sqm
→ доступность × инфраструктура на м²

f_int_access_x_office_ps = access_score * office_per_sqm
→ доступность × деловая активность вокруг

f_int_amen_over_access_total = amenities_total / (access_total_km + 1)
→ инфраструктура на единицу удалённости (

f_int_office_over_access_total = office_per_sqm / (access_total_km + 1)
→ деловая активность на удалённость

f_int_metro_close_x_amen = (metro<=1км)*amenities_total
→ инфраструктура при условии, что метро близко

f_int_mkad_close_x_amen = (mkad<=10км)*amenities_total
→ инфраструктура при условии, что близко к МКАД

3) Риск × доступность и удобства

f_int_risk_x_access = risk_index * access_total_km
→ плохо, когда и риск высокий, и далеко.

f_int_risk_over_access_score = risk_index / access_score
→ риск на доступность больше = хуже

f_int_risk_over_amen = risk_index / (amenities_total + 1)
→ риск на инфраструктуру

f_int_amen_over_risk = amenities_total / (risk_index + 1)
→ инфраструктура перекрывает риск больше = лучше

f_int_env_x_noise = env_risk_count * noise_risk_count
→ когда одновременно и экология плохая и шум.

f_int_risk_high_and_metro_close = 1 если risk_index>=2 и metro<=1км
→ метро рядом, но риск высокий

4) Дом возраст × этажность × качество планировки

f_int_age_x_floor_ratio = building_age * floor_ratio
→ эффект старый дом + высокий, низкий этаж

f_int_old_x_top_floor = 1 если дом старый и последний этаж
→ потенциально проблемная комбинация

f_int_new_x_not_first = 1 если дом новый и НЕ первый этаж


f_int_life_share_x_age = life_share * building_age
→ доля жилого связана с возрастом дома.

f_int_sqm_room_x_age = sqm_per_room * building_age
→ просторность × возраст.

5) Семья и дети × инфраструктура

f_int_childdens_x_schools = child_density * school_per_child
→ детская плотность + обеспеченность школами

f_int_family_index = school_per_child + 0.5*child_density + 0.1*amenities_total
→ общий индекс семейности района

f_int_family_index_over_risk = family_index / (risk_index + 1)
→ семейность с поправкой на риск

f_int_married_x_family_index = is_married * family_index
→ семейность района * актуальна* для женатых

6) Время × поведение при  сезонности

f_int_weekend_x_trans_amt_m = is_weekend * trans_amt_per_month
→ выходные × траты в месяц

f_int_weekend_x_trans_ct_m = is_weekend * trans_ct_per_month
→ выходные × частота транзакций

f_int_month_sin_x_trans_amt_m = month_sin * trans_amt_per_month
→ сезонность (sin) × траты

f_int_month_cos_x_trans_amt_m = month_cos * trans_amt_per_month
→ сезонность (cos) × траты

7) Нелинейные усилители (логи/квадраты)

f_int_log1p_amen = log(1 + amenities_total)
→ сжатая шкала инфраструктуры

f_int_log1p_access_total = log(1 + access_total_km)
→ сжатая шкала удалённости

f_int_log1p_risk = log(1 + risk_index)
→ сжатая шкала риска

f_int_risk_sq = (risk_index)^2
→ усиливает влияние больших рисков

f_int_access_score_sq = (access_score)^2
→ усиливает  хорошую доступность

f_int_amen_sq = (amenities_total)^2
→ усиливает  богатую инфраструктуру

8) Композитный индекс привлекательности района

f_int_attractiveness_index =

0.5log⁡(1+amen)+2⋅access_score−1.2log⁡(1+risk)+0.3⋅eco
0.5log(1+amen)+2⋅access_score−1.2log(1+risk)+0.3⋅eco

→ общий индекс - много инфраструктуры + хорошая доступность − риск + хорошая экология.


In [19]:
# Загрузка данных и первичная проверка
import pandas as pd

NAME = "Kirill_Viksman"
dataset = pd.read_csv(f"/content/0_{NAME}.csv", index_col=0)
SHAPE = dataset.shape

train = dataset[dataset["dtype"] == "train"]
test  = dataset[dataset["dtype"] == "test"]

SHAPE, train.shape, test.shape

((30471, 62), (20483, 62), (9988, 62))

In [20]:
# Загрузка данных и первичная проверка

df = pd.read_csv("/content/0_Kirill_Viksman.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", None)

df.head()

,Unnamed: 0,dtype,max_floor,state,marital_status,big_market_raion,total_revolving_bal,market_count_1500,leisure_count_3000,total_ct_chng_q4_q1,water_1line,railroad_station_walk_km,culture_objects_top_25,contacts_count_12_mon,0_17_all,trc_count_2000,product_type,build_count_wood,credit_limit,total_trans_ct,leisure_count_5000,life_sq,cafe_count_1000_price_1000,mkad_km,school_education_centers_top_20_raion,big_road1_1line,card_category,avg_utilization_ratio,public_transport_station_min_walk,income_category,customer_age,thermal_power_plant_raion,radiation_raion,detention_facility_km,sport_count_2000,cafe_sum_1000_min_price_avg,total_amt_chng_q4_q1,ecology,metro_km_walk,office_sqm_5000,gender,oil_chemistry_raion,nuclear_reactor_raion,total_trans_amt,months_inactive_12_mon,cafe_sum_1500_min_price_avg,railroad_1line,floor,num_room,timestamp,education_level,months_on_book,dependent_count,avg_open_to_buy,build_year,incineration_raion,full_sq,total_relationship_count,detention_facility_raion,build_count_mix,railroad_terminal_raion,__churn,__price
0,0,train,5.00,2.00,Married,no,866,0,3,0.78,no,1.12,no,2,3 771.00,4,Investment,0.00,3 279.90,43,8,16.00,1,8.58,0,no,Blue,0.10,3.45,$40K - $60K,38,no,no,7.17,10,966.67,0.77,poor,1.11,2916854,M,no,no,1 695.00,2,822.22,no,5.00,1.00,2013-09-13,Graduate,29,3,4 407.20,1 958.00,no,32,5,no,0.00,no,0.00,5.50
1,1,train,17.00,2.00,Single,no,7,2,0,0.64,no,5.79,no,2,12 962.00,6,Investment,0.00,12 951.00,64,1,30.00,2,1.58,0,no,Blue,0.07,2.98,Unknown,54,no,no,9.53,17,600.00,0.69,good,1.29,300476,M,no,no,6 694.00,3,657.14,no,11.00,2.00,2014-03-27,Graduate,36,2,18 155.90,2 005.00,no,52,3,no,0.00,no,0.00,7.00
2,2,train,NaN,NaN,Married,no,0,0,0,0.61,no,15.33,no,3,411.00,0,OwnerOccupier,NaN,1 438.30,21,0,NaN,0,12.69,0,no,Blue,0.41,52.61,Less than $40K,44,no,no,37.92,0,NaN,0.68,no data,13.31,0,F,no,no,1 522.00,2,NaN,no,3.00,NaN,2012-12-04,Graduate,36,2,1 232.20,NaN,no,42,3,no,NaN,no,1.00,3.33
3,3,train,1.00,1.00,Single,no,1370,0,0,0.50,no,1.92,no,2,3 831.00,0,OwnerOccupier,0.00,1 647.80,81,0,39.00,0,5.95,0,no,Blue,0.55,0.43,Less than $40K,50,no,no,12.24,0,NaN,0.63,good,3.78,26950,F,no,no,3 891.00,3,NaN,no,5.00,1.00,2013-10-29,Graduate,43,2,1 650.30,1.00,yes,39,2,no,0.00,no,0.00,4.14
4,4,train,9.00,2.00,Married,no,1382,3,4,0.56,no,NaN,no,3,NaN,10,Investment,16.00,2 136.00,61,23,18.00,5,10.49,0,yes,Blue,0.77,1.40,$40K - $60K,45,no,yes,4.07,13,861.11,1.14,poor,NaN,4768263,M,no,no,NaN,2,725.71,yes,6.00,1.00,2014-06-04,Graduate,46,3,1 258.20,1 970.00,no,32,5,no,2.00,no,0.00,6.15


In [21]:
# чеклист
import numpy as np

# Типы, пропуски
dtypes = df.dtypes
missing = df.isna().mean().sort_values(ascending=False)

display(dtypes.head(20))
display(missing.head(20))

# Базовая статистика числовых
display(df.select_dtypes(include=np.number).describe().T)

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))

df.select_dtypes(include="number").describe().T

num_desc = df.select_dtypes(include=np.number).describe().T

# Частоты для категориальных

cat_cols = df.select_dtypes(include="object").columns
for c in cat_cols:
    print("\n", c)
    display(df[c].value_counts(dropna=False).head(10))

,0
Unnamed: 0,int64
dtype,object
max_floor,float64
state,float64
marital_status,object
big_market_raion,object
total_revolving_bal,int64
market_count_1500,int64
leisure_count_3000,int64
total_ct_chng_q4_q1,float64


,0
build_year,0.45
state,0.44
0_17_all,0.41
total_trans_amt,0.41
__churn,0.33
__price,0.33
max_floor,0.31
num_room,0.31
cafe_sum_1000_min_price_avg,0.21
life_sq,0.21


,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,30 471.00,8 520.94,5 682.95,0.00,3 808.50,7 617.00,12 864.50,20 482.00
max_floor,20 931.00,12.51,6.73,0.00,9.00,12.00,17.00,117.00
state,16 959.00,2.10,0.85,1.00,1.00,2.00,3.00,4.00
total_revolving_bal,30 471.00,1 133.98,785.04,0.00,563.00,1 192.00,1 735.00,2 517.00
market_count_1500,30 471.00,0.76,1.11,0.00,0.00,0.00,1.00,7.00
leisure_count_3000,30 471.00,3.96,13.39,0.00,0.00,0.00,2.00,85.00
total_ct_chng_q4_q1,30 471.00,0.66,0.17,0.00,0.57,0.68,0.77,2.28
railroad_station_walk_km,30 446.00,4.37,3.82,0.06,1.93,3.23,5.13,24.65
contacts_count_12_mon,30 471.00,2.28,0.54,0.00,2.00,2.00,3.00,6.00
0_17_all,18 097.00,12 758.22,9 238.17,411.00,3 831.00,12 508.00,17 354.00,45 170.00



 dtype


,count
dtype,
train,20483
test,9988


...

,count
incineration_raion,
no,28132
yes,2339



 detention_facility_raion


,count
detention_facility_raion,
no,27414
yes,3057



 railroad_terminal_raion


,count
railroad_terminal_raion,
no,29317
yes,1154


# YOUR WORK START HERE

In [22]:
import numpy as np
import pandas as pd

new_dataset = dataset.copy()

# какие колонки не используем для генерации
exclude = {"dtype", "__price", "__churn"}

num_cols = [c for c in new_dataset.select_dtypes(include="number").columns if c not in exclude]
cat_cols = [c for c in new_dataset.select_dtypes(include="object").columns if c not in exclude]

features = {}

#  Числовые
for col in num_cols:
    s_all = new_dataset[col]
    s_tr  = train[col]

    # 1) индикатор пропуска
    features[f"f_isna_{col}"] = s_all.isna().astype("int8")

    # 2) imputing по train (median)
    med = s_tr.median()
    s_all_imp = s_all.fillna(med)
    s_tr_imp  = s_tr.fillna(med)

    # 3) clipping выбросов по train (1% / 99%)
    q01 = s_tr_imp.quantile(0.01)
    q99 = s_tr_imp.quantile(0.99)
    s_all_clip = s_all_imp.clip(q01, q99)
    features[f"f_clip_{col}"] = s_all_clip.astype("float32")

    # 4) robust scaling по train (медиана и IQR)
    s_tr_clip = s_tr_imp.clip(q01, q99)
    med_c = s_tr_clip.median()
    iqr = s_tr_clip.quantile(0.75) - s_tr_clip.quantile(0.25)
    if iqr == 0 or pd.isna(iqr):
        s_rs = (s_all_clip - med_c)
    else:
        s_rs = (s_all_clip - med_c) / iqr
    features[f"f_rs_{col}"] = s_rs.astype("float32")

    # 5) log1p
    if q01 >= 0:
        features[f"f_log1p_{col}"] = np.log1p(s_all_clip).astype("float32")

def safe_level(x: str, max_len=40):
    x = str(x)
    x = "".join(ch if ch.isalnum() else "_" for ch in x)
    return x[:max_len]

TOP_K = 3

for col in cat_cols:
    s_all = new_dataset[col].fillna("missing").astype(str)
    s_tr  = train[col].fillna("missing").astype(str)

    # 1) frequency encoding по train
    freq = s_tr.value_counts(normalize=True)
    features[f"f_freq_{col}"] = s_all.map(freq).fillna(0).astype("float32")

    # 2) one-hot только для топ-k категорий по train
    top_levels = freq.head(TOP_K).index.tolist()
    for lvl in top_levels:
        features[f"f_ohe_{col}__{safe_level(lvl)}"] = (s_all == lvl).astype("int8")

feat_df = pd.DataFrame(features, index=new_dataset.index)
new_dataset = pd.concat([new_dataset, feat_df], axis=1)

print("f_ features:", (new_dataset.columns.str.startswith("f_")).sum())
print("shape:", new_dataset.shape)

f_ features: 220
shape: (30471, 282)


In [23]:
import numpy as np
import pandas as pd

eps = 1e-6

def CL(col):
    name = f"f_clip_{col}"
    if name in new_dataset.columns:
        return new_dataset[name].astype(float)
    s = pd.to_numeric(new_dataset[col], errors="coerce")
    med = pd.to_numeric(train[col], errors="coerce").median()
    s = s.fillna(med)
    q01, q99 = s.loc[train.index].quantile(0.01), s.loc[train.index].quantile(0.99)
    return s.clip(q01, q99).astype(float)

def yn(col):
    # yes/no -> 1/0,
    if col not in new_dataset.columns:
        return pd.Series(0, index=new_dataset.index, dtype="int8")
    return (new_dataset[col].fillna("no").astype(str).str.lower() == "yes").astype("int8")

#  1) Недвижимость
full_sq = CL("full_sq")
life_sq = CL("life_sq")
num_room = CL("num_room")
floor = CL("floor")
max_floor = CL("max_floor")

new_dataset["f_sem_life_share"] = (life_sq / (full_sq + eps)).astype("float32")
new_dataset["f_sem_non_life_sq"] = (full_sq - life_sq).astype("float32")
new_dataset["f_sem_non_life_share"] = ((full_sq - life_sq) / (full_sq + eps)).astype("float32")

new_dataset["f_sem_sqm_per_room"] = (full_sq / (num_room + eps)).astype("float32")
new_dataset["f_sem_rooms_per_10sqm"] = (num_room / (full_sq / 10 + eps)).astype("float32")

new_dataset["f_sem_floor_ratio"] = (floor / (max_floor + eps)).astype("float32")
new_dataset["f_sem_floors_from_top"] = (max_floor - floor).astype("float32")
new_dataset["f_sem_is_first_floor"] = (floor.round() <= 1).astype("int8")
new_dataset["f_sem_is_top_floor"] = (floor.round() == max_floor.round()).astype("int8")
new_dataset["f_sem_is_low_floor"] = (floor.round() <= 2).astype("int8")
new_dataset["f_sem_is_high_floor"] = ((floor / (max_floor + eps)) >= 0.8).astype("int8")

# состояние дома
state = CL("state")
new_dataset["f_sem_state_low"] = (state <= 2).astype("int8")
new_dataset["f_sem_state_high"] = (state >= 3).astype("int8")

# возраст дома по timestamp - build_year
build_year = CL("build_year")
ts = pd.to_datetime(new_dataset["timestamp"], errors="coerce")
year = ts.dt.year.fillna(ts.dt.year.mode().iloc[0]).astype(int)
age = (year - build_year).clip(lower=0, upper=200)
new_dataset["f_sem_building_age"] = age.astype("float32")
new_dataset["f_sem_is_new_build"] = (age <= 5).astype("int8")
new_dataset["f_sem_is_old_build"] = (age >= 50).astype("int8")

# 2) Доступность и расстояния
mkad_km = CL("mkad_km")
metro_km_walk = CL("metro_km_walk")
rail_walk_km = CL("railroad_station_walk_km")
det_km = CL("detention_facility_km")
pt_walk = CL("public_transport_station_min_walk")

new_dataset["f_sem_access_total_km"] = (mkad_km + metro_km_walk + rail_walk_km + det_km).astype("float32")
new_dataset["f_sem_access_score"] = (
    1/(1+mkad_km) + 1/(1+metro_km_walk) + 1/(1+rail_walk_km)
).astype("float32")
new_dataset["f_sem_mkad_metro_ratio"] = (mkad_km / (metro_km_walk + eps)).astype("float32")
new_dataset["f_sem_log_pt_walk"] = np.log1p(pt_walk).astype("float32")

# 3) Инфраструктура
market = CL("market_count_1500")
trc = CL("trc_count_2000")
sport = CL("sport_count_2000")
leisure3 = CL("leisure_count_3000")
leisure5 = CL("leisure_count_5000")
cafe_cnt = CL("cafe_count_1000_price_1000")
office_sqm = CL("office_sqm_5000")
schools = CL("school_education_centers_top_20_raion")
kids017 = CL("0_17_all")

new_dataset["f_sem_leisure_total"] = (leisure3 + leisure5).astype("float32")
new_dataset["f_sem_amenities_total"] = (market + trc + sport + leisure3 + leisure5 + cafe_cnt + schools).astype("float32")
new_dataset["f_sem_amenities_per_sqm"] = (new_dataset["f_sem_amenities_total"] / (full_sq + eps)).astype("float32")
new_dataset["f_sem_office_per_sqm"] = (office_sqm / (full_sq + eps)).astype("float32")

new_dataset["f_sem_child_density"] = (kids017 / (full_sq + eps)).astype("float32")
new_dataset["f_sem_school_per_child"] = (schools / (kids017 + 1)).astype("float32")

# кафе-цены - отношение и разница
cafe1000 = CL("cafe_sum_1000_min_price_avg")
cafe1500 = CL("cafe_sum_1500_min_price_avg")
new_dataset["f_sem_cafe_price_ratio_1500_1000"] = (cafe1500 / (cafe1000 + eps)).astype("float32")
new_dataset["f_sem_cafe_price_diff_1500_1000"] = (cafe1500 - cafe1000).astype("float32")

# материалы застройки в долях
wood = CL("build_count_wood")
mix = CL("build_count_mix")
den = (wood + mix + 1.0)
new_dataset["f_sem_wood_share"] = (wood / den).astype("float32")
new_dataset["f_sem_mix_share"] = (mix / den).astype("float32")

# 4) Экология и риски
new_dataset["f_sem_water_near"] = yn("water_1line")
new_dataset["f_sem_big_market"] = yn("big_market_raion")

env_cols = ["oil_chemistry_raion","nuclear_reactor_raion","incineration_raion","radiation_raion","thermal_power_plant_raion"]
new_dataset["f_sem_env_risk_count"] = sum(yn(c) for c in env_cols).astype("int8")

noise_cols = ["big_road1_1line","railroad_1line"]
new_dataset["f_sem_noise_risk_count"] = sum(yn(c) for c in noise_cols).astype("int8")

new_dataset["f_sem_culture_top25"] = yn("culture_objects_top_25")
new_dataset["f_sem_detention_raion"] = yn("detention_facility_raion")
new_dataset["f_sem_rail_terminal_raion"] = yn("railroad_terminal_raion")

# суммарный риск-индекс (весами)
new_dataset["f_sem_risk_index"] = (
    2*new_dataset["f_sem_env_risk_count"]
    + 1*new_dataset["f_sem_noise_risk_count"]
    + 1*new_dataset["f_sem_detention_raion"]
).astype("float32")
new_dataset["f_sem_risk_x_access"] = (new_dataset["f_sem_risk_index"] * new_dataset["f_sem_access_total_km"]).astype("float32")

# 5) Банковские фитчи
credit_limit = CL("credit_limit")
open_to_buy = CL("avg_open_to_buy")
util = CL("avg_utilization_ratio")
revolve = CL("total_revolving_bal")

trans_amt = CL("total_trans_amt")
trans_ct = CL("total_trans_ct")
rel_cnt = CL("total_relationship_count")
months = CL("months_on_book")
inactive12 = CL("months_inactive_12_mon")
contacts12 = CL("contacts_count_12_mon")
age_cust = CL("customer_age")
dep = CL("dependent_count")

used = (credit_limit - open_to_buy).clip(lower=0)

new_dataset["f_sem_used_credit"] = used.astype("float32")
new_dataset["f_sem_open_to_buy_ratio"] = (open_to_buy / (credit_limit + eps)).astype("float32")
new_dataset["f_sem_used_credit_ratio"] = (used / (credit_limit + eps)).astype("float32")
new_dataset["f_sem_revolving_ratio"] = (revolve / (credit_limit + eps)).astype("float32")
new_dataset["f_sem_util_x_used_ratio"] = (util * new_dataset["f_sem_used_credit_ratio"]).astype("float32")

new_dataset["f_sem_trans_avg_amount"] = (trans_amt / (trans_ct + eps)).astype("float32")
new_dataset["f_sem_trans_amt_per_month"] = (trans_amt / (months + eps)).astype("float32")
new_dataset["f_sem_trans_ct_per_month"] = (trans_ct / (months + eps)).astype("float32")
new_dataset["f_sem_trans_amt_per_rel"] = (trans_amt / (rel_cnt + eps)).astype("float32")
new_dataset["f_sem_trans_ct_per_rel"] = (trans_ct / (rel_cnt + eps)).astype("float32")

new_dataset["f_sem_contacts_per_month"] = (contacts12 / (months + eps)).astype("float32")
new_dataset["f_sem_inactive_ratio"] = (inactive12 / (months + eps)).clip(0, 1).astype("float32")
new_dataset["f_sem_active_ratio"] = (1 - new_dataset["f_sem_inactive_ratio"]).astype("float32")

new_dataset["f_sem_rel_per_age"] = (rel_cnt / (age_cust + 1)).astype("float32")
new_dataset["f_sem_dep_ratio"] = (dep / (age_cust + 1)).astype("float32")

amt_ch = CL("total_amt_chng_q4_q1")
ct_ch = CL("total_ct_chng_q4_q1")
new_dataset["f_sem_change_interaction"] = (amt_ch * ct_ch).astype("float32")
new_dataset["f_sem_change_ratio"] = (amt_ch / (ct_ch + eps)).astype("float32")

# utilisation флаги
new_dataset["f_sem_util_low"] = (util < 0.2).astype("int8")
new_dataset["f_sem_util_high"] = (util > 0.8).astype("int8")

#  6) Время из timestamp (сезонность)
month = ts.dt.month.fillna(6).astype(int)
dow = ts.dt.dayofweek.fillna(0).astype(int)

new_dataset["f_sem_year"] = year.astype("int16")
new_dataset["f_sem_month"] = month.astype("int8")
new_dataset["f_sem_quarter"] = ts.dt.quarter.fillna(2).astype(int).astype("int8")
new_dataset["f_sem_dayofweek"] = dow.astype("int8")
new_dataset["f_sem_is_weekend"] = (dow >= 5).astype("int8")

m = month.astype(float)
new_dataset["f_sem_month_sin"] = np.sin(2*np.pi*m/12).astype("float32")
new_dataset["f_sem_month_cos"] = np.cos(2*np.pi*m/12).astype("float32")

d = dow.astype(float)
new_dataset["f_sem_dow_sin"] = np.sin(2*np.pi*d/7).astype("float32")
new_dataset["f_sem_dow_cos"] = np.cos(2*np.pi*d/7).astype("float32")

# 7) Ординальные  ранги категорий
income_map = {
    "Less than $40K": 0,
    "$40K - $60K": 1,
    "$60K - $80K": 2,
    "$80K - $120K": 3,
    "$120K +": 4,
    "Unknown": 2
}
card_map = {"Blue": 0, "Silver": 1, "Gold": 2}
eco_map = {"poor": 0, "satisfactory": 1, "no data": 2, "good": 3, "excellent": 4}
edu_map = {"High School": 0, "Graduate": 1, "Unknown": 1}

new_dataset["f_sem_income_rank"] = new_dataset["income_category"].map(income_map).fillna(2).astype("int8")
new_dataset["f_sem_card_rank"] = new_dataset["card_category"].map(card_map).fillna(0).astype("int8")
new_dataset["f_sem_ecology_rank"] = new_dataset["ecology"].map(eco_map).fillna(2).astype("int8")
new_dataset["f_sem_education_rank"] = new_dataset["education_level"].map(edu_map).fillna(1).astype("int8")

new_dataset["f_sem_is_male"] = new_dataset["gender"].astype(str).str.upper().eq("M").astype("int8")
new_dataset["f_sem_is_married"] = new_dataset["marital_status"].astype(str).str.lower().eq("married").astype("int8")
new_dataset["f_sem_is_investment"] = new_dataset["product_type"].astype(str).str.lower().eq("investment").astype("int8")

#  Проверка
sem_cols = [c for c in new_dataset.columns if c.startswith("f_sem_")]
print("Добавили f_sem_:", len(sem_cols))
print("NaN в f_sem_:", int(new_dataset[sem_cols].isna().sum().sum()))
print("Все f_sem_ числовые:", all(pd.api.types.is_numeric_dtype(new_dataset[c]) for c in sem_cols))

Добавили f_sem_: 74
NaN в f_sem_: 0
Все f_sem_ числовые: True


In [24]:
import numpy as np
import pandas as pd

eps = 1e-6

def S(col, default=0.0):
    """Безопасно вернуть колонку как float (если нет — константа)."""
    if col in new_dataset.columns:
        return pd.to_numeric(new_dataset[col], errors="coerce").fillna(default).astype(float)
    return pd.Series(default, index=new_dataset.index, dtype=float)

def clip01(x):
    return np.clip(x, 0, 1)

# Берём уже созданные смысловые признаки (из f_sem_ блока)
income = S("f_sem_income_rank")
card = S("f_sem_card_rank")
eco = S("f_sem_ecology_rank")

used_ratio = S("f_sem_used_credit_ratio")
open_ratio = S("f_sem_open_to_buy_ratio")
util_low = S("f_sem_util_low")
util_high = S("f_sem_util_high")

trans_amt_m = S("f_sem_trans_amt_per_month")
trans_ct_m = S("f_sem_trans_ct_per_month")
trans_avg = S("f_sem_trans_avg_amount")

amen = S("f_sem_amenities_total")
amen_ps = S("f_sem_amenities_per_sqm")
office_ps = S("f_sem_office_per_sqm")

access_total = S("f_sem_access_total_km")
access_score = S("f_sem_access_score")
mkad = S("f_clip_mkad_km") if "f_clip_mkad_km" in new_dataset.columns else S("mkad_km")
metro = S("f_clip_metro_km_walk") if "f_clip_metro_km_walk" in new_dataset.columns else S("metro_km_walk")

risk = S("f_sem_risk_index")
env_risk = S("f_sem_env_risk_count")
noise_risk = S("f_sem_noise_risk_count")

age_b = S("f_sem_building_age")
new_build = S("f_sem_is_new_build")
old_build = S("f_sem_is_old_build")

floor_ratio = S("f_sem_floor_ratio")
top_floor = S("f_sem_is_top_floor")
first_floor = S("f_sem_is_first_floor")

life_share = S("f_sem_life_share")
sqm_room = S("f_sem_sqm_per_room")
rooms10 = S("f_sem_rooms_per_10sqm")

child_dens = S("f_sem_child_density")
school_pc = S("f_sem_school_per_child")

is_weekend = S("f_sem_is_weekend")
month_sin = S("f_sem_month_sin")
month_cos = S("f_sem_month_cos")

is_male = S("f_sem_is_male")
is_married = S("f_sem_is_married")


# 1) взаимодействия доход и карта × финансы
new_dataset["f_int_income_x_used_ratio"] = (income * used_ratio).astype("float32")
new_dataset["f_int_income_x_open_ratio"] = (income * open_ratio).astype("float32")
new_dataset["f_int_income_x_trans_amt_m"] = (income * trans_amt_m).astype("float32")
new_dataset["f_int_income_x_trans_ct_m"] = (income * trans_ct_m).astype("float32")
new_dataset["f_int_income_x_trans_avg"] = (income * trans_avg).astype("float32")

new_dataset["f_int_card_x_used_ratio"] = (card * used_ratio).astype("float32")
new_dataset["f_int_card_x_trans_amt_m"] = (card * trans_amt_m).astype("float32")
new_dataset["f_int_card_x_trans_ct_m"] = (card * trans_ct_m).astype("float32")

new_dataset["f_int_high_util_and_high_income"] = ((util_high > 0) & (income >= 3)).astype("int8")
new_dataset["f_int_low_util_and_low_income"] = ((util_low > 0) & (income <= 1)).astype("int8")


# 2) Инфраструктура × доступность
new_dataset["f_int_access_x_amen"] = (access_score * amen).astype("float32")
new_dataset["f_int_access_x_amen_ps"] = (access_score * amen_ps).astype("float32")
new_dataset["f_int_access_x_office_ps"] = (access_score * office_ps).astype("float32")

new_dataset["f_int_amen_over_access_total"] = (amen / (access_total + 1.0)).astype("float32")
new_dataset["f_int_office_over_access_total"] = (office_ps / (access_total + 1.0)).astype("float32")

new_dataset["f_int_metro_close_x_amen"] = ((metro <= 1.0).astype("int8") * amen).astype("float32")
new_dataset["f_int_mkad_close_x_amen"] = ((mkad <= 10.0).astype("int8") * amen).astype("float32")


# 3) Риск × доступность и  удобства
new_dataset["f_int_risk_x_access"] = (risk * access_total).astype("float32")
new_dataset["f_int_risk_over_access_score"] = (risk / (access_score + eps)).astype("float32")
new_dataset["f_int_risk_over_amen"] = (risk / (amen + 1.0)).astype("float32")
new_dataset["f_int_amen_over_risk"] = (amen / (risk + 1.0)).astype("float32")

new_dataset["f_int_env_x_noise"] = (env_risk * noise_risk).astype("float32")
new_dataset["f_int_risk_high_and_metro_close"] = ((risk >= 2).astype("int8") & (metro <= 1.0)).astype("int8")


# 4) Дом возраст × этажность × качество планировки
new_dataset["f_int_age_x_floor_ratio"] = (age_b * floor_ratio).astype("float32")
new_dataset["f_int_old_x_top_floor"] = ((old_build > 0) & (top_floor > 0)).astype("int8")
new_dataset["f_int_new_x_not_first"] = ((new_build > 0) & (first_floor == 0)).astype("int8")

new_dataset["f_int_life_share_x_age"] = (life_share * age_b).astype("float32")
new_dataset["f_int_sqm_room_x_age"] = (sqm_room * age_b).astype("float32")


# 5) Семьи и дети × инфраструктура
new_dataset["f_int_childdens_x_schools"] = (child_dens * school_pc).astype("float32")
new_dataset["f_int_family_index"] = (school_pc + 0.5*child_dens + 0.1*amen).astype("float32")
new_dataset["f_int_family_index_over_risk"] = (new_dataset["f_int_family_index"] / (risk + 1.0)).astype("float32")

new_dataset["f_int_married_x_family_index"] = (is_married * new_dataset["f_int_family_index"]).astype("float32")


# 6) ВРЕМЯ × поведение при сезонности
new_dataset["f_int_weekend_x_trans_amt_m"] = (is_weekend * trans_amt_m).astype("float32")
new_dataset["f_int_weekend_x_trans_ct_m"] = (is_weekend * trans_ct_m).astype("float32")

new_dataset["f_int_month_sin_x_trans_amt_m"] = (month_sin * trans_amt_m).astype("float32")
new_dataset["f_int_month_cos_x_trans_amt_m"] = (month_cos * trans_amt_m).astype("float32")


# 7) Нелинейные  усилители квадраты и логи индексов
new_dataset["f_int_log1p_amen"] = np.log1p(amen).astype("float32")
new_dataset["f_int_log1p_access_total"] = np.log1p(access_total).astype("float32")
new_dataset["f_int_log1p_risk"] = np.log1p(risk).astype("float32")

new_dataset["f_int_risk_sq"] = (risk**2).astype("float32")
new_dataset["f_int_access_score_sq"] = (access_score**2).astype("float32")
new_dataset["f_int_amen_sq"] = (amen**2).astype("float32")


# 8) Композитный индекс привлекательности
new_dataset["f_int_attractiveness_index"] = (
    0.5*np.log1p(amen) + 2.0*access_score - 1.2*np.log1p(risk) + 0.3*eco
).astype("float32")


# Проверки

int_cols = [c for c in new_dataset.columns if c.startswith("f_int_")]
print("Добавили f_int_:", len(int_cols))
print("NaN в f_int_:", int(new_dataset[int_cols].isna().sum().sum()))
print("Все f_int_ числовые:", all(pd.api.types.is_numeric_dtype(new_dataset[c]) for c in int_cols))

Добавили f_int_: 43
NaN в f_int_: 0
Все f_int_ числовые: True


/tmp/ipython-input-263/4079848897.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_dataset["f_int_env_x_noise"] = (env_risk * noise_risk).astype("float32")
/tmp/ipython-input-263/4079848897.py:99: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_dataset["f_int_risk_high_and_metro_close"] = ((risk >= 2).astype("int8") & (metro <= 1.0)).astype("int8")
/tmp/ipython-input-263/4079848897.py:103: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

# YOUR WORK IS DONE (Create submission)

In [25]:
if new_dataset.shape[0] != SHAPE[0]:
    raise ValueError(f'Incorrect OUTPUT file shape. Original {SHAPE[0]}. {dataset.shape[0]} are given')

num_new_features = new_dataset.columns[new_dataset.columns.str.startswith("f_")].shape[0]
if num_new_features < 50:
    raise ValueError(f'You need 50 new features. {num_new_features} features are given')

# index must be True
output_path = '1_' + NAME + '.csv'

print(f"output file {output_path} was created")
new_dataset.to_csv(output_path, index=True)

output file 1_Kirill_Viksman.csv was created


In [26]:
# Проверка
f_cols = [c for c in new_dataset.columns if c.startswith("f_")]
print("f_ count:", len(f_cols))
print("missing in f_:", int(new_dataset[f_cols].isna().sum().sum()))
print("all f_ numeric:", all(pd.api.types.is_numeric_dtype(new_dataset[c]) for c in f_cols))

f_ count: 337
missing in f_: 0
all f_ numeric: True


In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/1_Kirill_Viksman.csv", index_col=0)

f_cols = [c for c in df.columns if c.startswith("f_")]

print("shape:", df.shape)
print("f_ count:", len(f_cols))
print("missing in f_:", int(df[f_cols].isna().sum().sum()))
print("all f_ numeric:", all(pd.api.types.is_numeric_dtype(df[c]) for c in f_cols))
print("dtype counts:", df["dtype"].value_counts().to_dict())

# распределение по группам
groups = {
    "isna": sum(c.startswith("f_isna_") for c in f_cols),
    "clip": sum(c.startswith("f_clip_") for c in f_cols),
    "rs": sum(c.startswith("f_rs_") for c in f_cols),
    "log1p": sum(c.startswith("f_log1p_") for c in f_cols),
    "freq": sum(c.startswith("f_freq_") for c in f_cols),
    "ohe": sum(c.startswith("f_ohe_") for c in f_cols),
    "sem": sum(c.startswith("f_sem_") for c in f_cols),
    "int": sum(c.startswith("f_int_") for c in f_cols),
}
print("groups:", groups)

shape: (30471, 399)
f_ count: 337
missing in f_: 0
all f_ numeric: True
dtype counts: {'train': 20483, 'test': 9988}
groups: {'isna': 39, 'clip': 39, 'rs': 39, 'log1p': 39, 'freq': 20, 'ohe': 44, 'sem': 74, 'int': 43}
